# hai07 - ukur efisiensi pelatihan (tahap B)
Input (Add Input): dataset **hai07-bench** (unggah `kaggle/hai07-bench.zip`; dibuat oleh `python tools/package_bench_kit.py`, ±1 MB).
Settings: Accelerator = **GPU T4** (satu GPU cukup; kode latih kita memakai satu GPU), Internet tidak perlu.
Tiap konfigurasi dijalankan sebagai proses terpisah (kehabisan memori tidak menghentikan yang lain), ±1 sampai 3 menit per konfigurasi.
Hasil: tabel token/detik, MFU, memori puncak, dan jam untuk 10 miliar token. **Kirim tabel itu kembali**: itulah dasar estimasi biaya dari angka terukur.
Catatan: `torch.compile` baru pertama kali dicoba di T4 di sini; bila gagal, baris itu menampilkan galatnya dan yang lain tetap jalan.

In [ ]:
import glob, os, subprocess, sys, zipfile, json
WORK = "/kaggle/working/hai07-bench"; os.makedirs(WORK, exist_ok=True)
src_dir = next((os.path.dirname(os.path.dirname(p)) for p in glob.glob("/kaggle/input/**/tools/bench_train.py", recursive=True)), None)
if src_dir: subprocess.run(["cp", "-r", src_dir + "/.", WORK], check=True)
else:
    z = glob.glob("/kaggle/input/**/hai07-bench.zip", recursive=True); assert z, "tambahkan dataset hai07-bench sebagai Input"
    zipfile.ZipFile(z[0]).extractall(WORK)
os.chdir(WORK)
import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "TIDAK ADA")
assert torch.cuda.is_available(), "aktifkan Accelerator GPU"


In [ ]:
# (nama, argumen). Konfigurasi pertama meniru pelatihan v2 (model 100M, konteks 256, batch 32, AMP) sebagai garis dasar.
RUNS = [
    ("100M garis dasar (seperti v2)",            ["--preset", "100m", "--ctx", "256", "--batch", "32", "--amp"]),
    ("100M + fused AdamW",                       ["--preset", "100m", "--ctx", "256", "--batch", "32", "--amp", "--fused-adamw"]),
    ("100M + fused + compile",                   ["--preset", "100m", "--ctx", "256", "--batch", "32", "--amp", "--fused-adamw", "--compile"]),
    ("100M konteks 1024, batch 8",               ["--preset", "100m", "--ctx", "1024", "--batch", "8", "--amp", "--fused-adamw"]),
    ("300M konteks 1024, ckpt, batch 4",         ["--preset", "300m", "--ctx", "1024", "--batch", "4", "--amp", "--fused-adamw", "--grad-checkpoint"]),
    ("540M konteks 1024, ckpt, batch 2",         ["--preset", "540m", "--ctx", "1024", "--batch", "2", "--amp", "--fused-adamw", "--grad-checkpoint"]),
    ("540M konteks 1024, ckpt, batch 1",         ["--preset", "540m", "--ctx", "1024", "--batch", "1", "--grad-accum", "2", "--amp", "--fused-adamw", "--grad-checkpoint"]),
    ("1B konteks 1024, ckpt, batch 1 (diduga OOM di T4)", ["--preset", "1b", "--ctx", "1024", "--batch", "1", "--amp", "--fused-adamw", "--grad-checkpoint"]),
]
OUT = "/kaggle/working/bench.jsonl"
if os.path.exists(OUT): os.remove(OUT)
rows = []
for name, extra in RUNS:
    print("=" * 90); print(name)
    p = subprocess.run([sys.executable, "-u", "tools/bench_train.py", "--device", "cuda", "--steps", "8", "--warmup", "3", "--out", OUT] + extra,
                       capture_output=True, text=True)
    last = [l for l in p.stdout.splitlines() if l.startswith("{")]
    if last:
        d = json.loads(last[-1]); d["nama"] = name; rows.append(d); print(last[-1])
    else:
        err = (p.stderr.strip().splitlines() or ["(tanpa keluaran)"])[-1][:200]
        rows.append({"nama": name, "galat": err}); print("GALAT:", err)


In [ ]:
cols = ["nama", "parameter_juta", "token_per_detik", "mfu_persen", "tflops_efektif", "memori_puncak_gb", "jam_untuk_10_miliar_token", "galat"]
w = {c: max(len(c), *(len(str(r.get(c, "-"))) for r in rows)) for c in cols}
print("  ".join(c.ljust(w[c]) for c in cols)); print("  ".join("-" * w[c] for c in cols))
for r in rows: print("  ".join(str(r.get(c, "-")).ljust(w[c]) for c in cols))
base = next((r for r in rows if "garis dasar" in r["nama"] and "token_per_detik" in r), None)
if base:
    print("\nGaris dasar:", base["token_per_detik"], "token/dtk, MFU", base["mfu_persen"], "% di", base["perangkat"])
    for r in rows:
        if "token_per_detik" in r and r is not base and r["preset"] == "100m" and r["konteks"] == 256:
            print(f"  {r['nama']}: {r['token_per_detik'] / base['token_per_detik']:.2f}x garis dasar (syarat gerbang B: >= 2x)")
print("\nSalin tabel ini dan kirim ke pengembang. Berkas lengkap:", OUT)


## Cara membaca
- **MFU** = berapa persen dari puncak teoretis GPU yang terpakai; model bahasa yang efisien biasanya 30 sampai 50% di GPU modern. Nilai satu digit berarti masih banyak waktu terbuang.
- **Gerbang B:** salah satu kombinasi pada model 100M harus mencapai ≥ 2× garis dasar. Kombinasi itu yang dipakai untuk model besar.
- **Memori puncak** menentukan apakah 300M atau 540M muat di GPU yang dipakai; baris dengan galat OOM berarti tidak muat (1B memang diperkirakan tidak muat di T4).
- Angka T4 hanya patokan relatif. Di A100 atau H100 efisiensi dan kecepatan berbeda; ukur ulang di sana sebelum mengunci anggaran.